In [35]:
print('hello world')

hello world


In [36]:
import docx
import json

In [37]:
doc = docx.Document('F:/ai_rag_based_chatbot/customer-support-rag-chat-system/resource/documents/global_shipping_policy.docx')

In [38]:
def extract_tag_content(line, tag):

    opening = f"<{tag}>"
    closing = f"</{tag}>"

    return (
        line.replace(opening, "")
            .replace(closing, "")
            .strip()
    )

In [39]:
lines = [para.text for para in doc.paragraphs]
lines[:4]

['<t> Global Shipping Policy </t>',
 '<h1>Overview</h1>',
 '<p>This policy defines shipping timelines, shipping fees, international shipping eligibility, shipment tracking procedures, and lost shipment handling for all customer orders placed through ABC Ecommerce.</p>',
 '<h1>Eligibility</h1>']

In [40]:
nodes = []
inside_paragraph = False
paragraph_buffer = []

for line in lines:

    line = line.strip()
    # blank line
    if not line:
        continue
    # actual line
    if inside_paragraph:

        if line.endswith("</p>"):
            
            clean = line.replace("</p>","")
            if clean:
                paragraph_buffer.append(clean)
            text = "\n".join(paragraph_buffer)
            nodes.append({
                        "type": "paragraph",
                        "text": text
                        })
            inside_paragraph = False
            paragraph_buffer = []
            
        else:
            paragraph_buffer.append(line)
            continue

            


    if line.startswith("<t>"):

      document_name = extract_tag_content(line, "t")
      nodes.append({
            "type": "title",
            "level": 1,
            "text": extract_tag_content(line, "t")
        })
            

    elif line.startswith("<h1>"):

        nodes.append({
            "type": "heading",
            "level": 1,
            "text": extract_tag_content(line, "h1")
        })

    elif line.startswith("<h2>"):
        nodes.append({
            "type" : "subheading",
            "level": 2,
            "text" : extract_tag_content(line, "h2")
        })

    elif line.startswith("<p>"):
        inside_paragraph = True
        

        if line.endswith("</p>"):
            
            inside_paragraph = False
            nodes.append({
                "type": "paragraph",
                "text": extract_tag_content(line, "p")
            })
        else:
            clean = line.replace("<p>","").strip()
            if clean:
                paragraph_buffer.append(clean)
        
        continue


    elif line.startswith("<li>"):

        nodes.append({
            "type": "list_item",
            "text": extract_tag_content(line, "li")
        })

    elif line.startswith("<faq-q>"):

        nodes.append({
            "type": "faq_question",
            "text": extract_tag_content(line, "faq-q")
        })

    elif line.startswith("<faq-a>"):

        nodes.append({
            "type": "faq_answer",
            "text": extract_tag_content(line, "faq-a")
        })

json_doc = {'document_name': document_name,
            'nodes': nodes}

json_output = json.dumps(json_doc, indent=2)
    

    

In [41]:
print(json_output)

{
  "document_name": "Global Shipping Policy",
  "nodes": [
    {
      "type": "title",
      "level": 1,
      "text": "Global Shipping Policy"
    },
    {
      "type": "heading",
      "level": 1,
      "text": "Overview"
    },
    {
      "type": "paragraph",
      "text": "This policy defines shipping timelines, shipping fees, international shipping eligibility, shipment tracking procedures, and lost shipment handling for all customer orders placed through ABC Ecommerce."
    },
    {
      "type": "heading",
      "level": 1,
      "text": "Eligibility"
    },
    {
      "type": "list_item",
      "text": "Applies to all products sold on the platform."
    },
    {
      "type": "list_item",
      "text": "Shipping availability depends on customer location."
    },
    {
      "type": "list_item",
      "text": "Certain regions may have restricted delivery."
    },
    {
      "type": "list_item",
      "text": "International shipping availability varies by product category."